# Pilot Model v4 — tuned LightGBM и secondary XGBoost

**Pilot Model v4** делает final fit на всей известной истории Jan–Oct 2025 и строит два отдельных Nov–Dec submission. Основная модель — tuned LightGBM, выбранная после [model selection](../../ml/experiments/results/model_selection.json) и [HPO](../../ml/experiments/results/hyperparameter_tuning.json) по pooled WAPE трёх temporal folds. Tuned XGBoost остаётся secondary candidate для возможного отдельного ensemble experiment. В этом notebook нет нового feature experiment, model selection, HPO или blending.

Обе модели используют один набор `SET_FEATURES + CALENDAR_FEATURES + WEATHER_FEATURES`: именно эта `C+W` конфигурация прошла предыдущие этапы. Accepted feature registry остаётся прежним; calendar и weather columns здесь — проверенная pilot configuration, а не изменение registry.

## Environment & Data Setup

Напрямую читаем два актуальных model-ready CSV. Jan–Oct содержит известный target `boardings`, включая Sep–Oct: этот период служил pseudo-future validation на предыдущих этапах, но для final fit уже входит в доступную историю. Nov–Dec содержит только известные заранее признаки и исходный порядок forecast grid. Labels, grid и признаки из raw не пересобираем.

In [1]:
from pathlib import Path
from time import perf_counter
import json
import numpy as np
import pandas as pd
from IPython.display import display
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'ml/src/model_selection.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Project root not found')
sys.path.insert(0, str(ROOT / 'ml/src'))
import model_selection as ms

DATA_ROOT = ROOT / 'data'
TRAIN_PATH = DATA_ROOT / 'processed/model_ready_train_2025_jan_oct.csv'
FORECAST_PATH = DATA_ROOT / 'processed/model_ready_forecast_2025_nov_dec.csv'
MODEL_SELECTION_PATH = ROOT / 'ml/experiments/results/model_selection.json'
HPO_PATH = ROOT / 'ml/experiments/results/hyperparameter_tuning.json'
SUBMISSION_PATHS = {
    'lightgbm_primary': DATA_ROOT / 'submissions/pilot_model_v4_lightgbm_submission.csv',
    'xgboost_secondary': DATA_ROOT / 'submissions/pilot_model_v4_xgboost_submission.csv',
}
BUCKET_DTYPE = ms.BUCKET_DTYPE
train = pd.read_csv(TRAIN_PATH, sep=';', encoding='utf-8', parse_dates=['date'],
                    dtype={'temperature_bucket': BUCKET_DTYPE})
forecast = pd.read_csv(FORECAST_PATH, sep=';', encoding='utf-8', parse_dates=['date'],
                       dtype={'temperature_bucket': BUCKET_DTYPE})
model_selection = json.loads(MODEL_SELECTION_PATH.read_text(encoding='utf-8'))
hpo = json.loads(HPO_PATH.read_text(encoding='utf-8'))
display(pd.DataFrame([{'train_rows': len(train), 'forecast_rows': len(forecast),
                       'train_last_date': train.date.max().date(),
                       'forecast_first_date': forecast.date.min().date(),
                       'forecast_last_date': forecast.date.max().date()}]))

,train_rows,forecast_rows,train_last_date,forecast_first_date,forecast_last_date
0,72960,14640,2025-10-31,2025-11-01,2025-12-31


## Data Contract / Sanity Checks

Проверяем тот же model-ready контракт, что в Pilot v3: точные схемы, строки, уникальный `(route,date,hour)`, все 10 маршрутов и 24 часа, полноту периода, отсутствие NaN и согласованность `temperature_bucket` с frozen climatology. Дополнительно проверяем, что forecast не содержит `boardings`. Ошибка здесь останавливает final fit.

In [2]:
ROUTES = ms.ROUTES
KEYS = ['route', 'date', 'hour']
SET_FEATURES = ms.SET_FEATURES
CALENDAR_FEATURES = ms.CALENDAR_FEATURES
WEATHER_FEATURES = ms.WEATHER_FEATURES
CALENDAR_WEATHER_FEATURES = SET_FEATURES + CALENDAR_FEATURES + WEATHER_FEATURES
BUCKET_LABELS = tuple(BUCKET_DTYPE.categories)

WEATHER_COLUMNS = (
    'climatological_temperature', 'climatological_precipitation',
    'climatological_rain_probability', 'temperature_bucket'
)
BUCKET_LABELS = ('<0', '[0,10)', '[10,20)', '>=20')
FORECAST_COLUMNS = [
    'route', 'date', 'hour', 'weekday', 'route_hour', 'hour_weekend',
    'is_night', *CALENDAR_FEATURES, *WEATHER_COLUMNS
]
TRAIN_COLUMNS = FORECAST_COLUMNS[:-len(WEATHER_COLUMNS)] + ['boardings', *WEATHER_COLUMNS]

if train.columns.tolist() != TRAIN_COLUMNS:
    raise ValueError('Train schema differs from model-ready contract')
if forecast.columns.tolist() != FORECAST_COLUMNS:
    raise ValueError('Forecast schema differs from model-ready contract')
if train.drop(columns='boardings').columns.tolist() != forecast.columns.tolist():
    raise ValueError('Train and forecast feature columns differ')
if set(train.columns) - set(KEYS) - {'boardings'} != set(SET_FEATURES + CALENDAR_FEATURES + WEATHER_COLUMNS) - set(KEYS):
    raise ValueError('Unexpected feature columns')

def check_grid(frame, start, end, expected_rows):
    dates = pd.date_range(start, end, freq='D')
    if len(frame) != expected_rows or frame[KEYS].isna().any().any():
        raise ValueError('Incorrect row count or missing keys')
    if frame.isna().any().any():
        raise ValueError('Missing values in model-ready data')
    if not pd.api.types.is_datetime64_any_dtype(frame['date']):
        raise TypeError('date must be datetime')
    if frame[KEYS].duplicated().any():
        raise ValueError('Duplicate route/date/hour keys')
    if set(frame['route']) != set(ROUTES) or set(frame['hour']) != set(range(24)):
        raise ValueError('Unexpected route or hour values')
    if set(frame['date']) != set(dates):
        raise ValueError('Incomplete or unexpected date period')
    if len(frame) != len(ROUTES) * len(dates) * 24:
        raise ValueError('Incomplete route/date/hour grid')
    if not all(pd.api.types.is_numeric_dtype(frame[col]) for col in frame.columns if col not in ('date', 'temperature_bucket')):
        raise TypeError('Non-numeric feature or target')
    if frame['temperature_bucket'].dtype != BUCKET_DTYPE:
        raise TypeError('Temperature bucket categorical dtype changed')
    if not frame['temperature_bucket'].isin(BUCKET_LABELS).all():
        raise ValueError('Unknown temperature bucket')
    expected_bucket = pd.cut(
        frame['climatological_temperature'], bins=[-np.inf, 0, 10, 20, np.inf],
        right=False, labels=BUCKET_LABELS,
    )
    if not frame['temperature_bucket'].eq(expected_bucket).all():
        raise ValueError('Temperature bucket differs from frozen climatology')
    if not frame['weekday'].eq(frame['date'].dt.dayofweek).all():
        raise ValueError('weekday does not match date')
    if not frame['route_hour'].eq(frame['route'] * 24 + frame['hour']).all():
        raise ValueError('route_hour encoding differs from v1')
    if not frame['hour_weekend'].eq(frame['hour'] * 2 + frame['weekday'].ge(5).astype('int8')).all():
        raise ValueError('hour_weekend encoding differs from v1')
    if not frame['is_night'].eq(frame['hour'].between(0, 5).astype('int8')).all():
        raise ValueError('is_night encoding differs from v1')
    for col in ('is_night', *CALENDAR_FEATURES):
        if not frame[col].isin((0, 1)).all():
            raise ValueError(f'{col} must be binary')

check_grid(train, '2025-01-01', '2025-10-31', 72_960)
check_grid(forecast, '2025-11-01', '2025-12-31', 14_640)
if 'boardings' in forecast.columns:
    raise ValueError('Forecast must not contain target')
if train['boardings'].lt(0).any():
    raise ValueError('Negative training target')
print('Model-ready data contract passed.')
display(pd.DataFrame([{'check': 'model-ready contract', 'status': 'passed',
                       'train_rows': len(train), 'forecast_rows': len(forecast),
                       'routes': forecast.route.nunique(), 'hours': forecast.hour.nunique()}]))

Model-ready data contract passed.


,check,status,train_rows,forecast_rows,routes,hours
0,model-ready contract,passed,72960,14640,10,24


## Feature Configuration & HPO Checkpoint

В модели входят только 12 уже определённых `C+W` columns; три непрерывных climatology columns остаются в CSV для контракта, но не подаются модели. `model_selection.json` и `hyperparameter_tuning.json` фиксируют выбор **до** Nov–Dec inference. Основной LightGBM получил pooled WAPE `0.150956`, secondary XGBoost — `0.153481`. В HPO Nov–Dec не использовался: target за этот период отсутствует, и оценить accuracy по будущим прогнозам нельзя.

Перед обучением сверяем feature list, данные, выбранные семьи и **полные** параметры с сохранёнными HPO artifacts. Categorical preprocessing берём из `model_selection.make_features`, как в HPO: фиксированные pandas categories; XGBoost использует native categorical splits, LightGBM — свой categorical feature contract.

In [3]:
CONFIGURATION = 'calendar_weather'
if CALENDAR_WEATHER_FEATURES != ms.FEATURE_CONFIGURATIONS[CONFIGURATION]:
    raise ValueError('C+W feature list changed')
if model_selection['features'][CONFIGURATION] != list(CALENDAR_WEATHER_FEATURES):
    raise ValueError('Model-selection feature list changed')
if hpo['configuration'] != CONFIGURATION or hpo['features'] != list(CALENDAR_WEATHER_FEATURES):
    raise ValueError('HPO feature configuration changed')
if model_selection['hpo_candidates'] != [
    {'family': 'XGBRegressor', 'configuration': CONFIGURATION},
    {'family': 'LightGBM', 'configuration': CONFIGURATION}
]:
    raise ValueError('Model-selection candidates changed')
if hpo['nov_dec_used'] or hpo['accepted_feature_registry_changed']:
    raise ValueError('HPO provenance contract changed')
for path in (TRAIN_PATH, MODEL_SELECTION_PATH, ROOT / 'ml/src/model_selection.py', ROOT / 'ml/src/hyperparameter_tuning.py'):
    relative = path.relative_to(ROOT).as_posix()
    if ms.sha256(path) != hpo['source_sha256'][relative]:
        raise ValueError(f'Source hash changed: {relative}')

LIGHTGBM_PARAMS = {**ms.MODEL_PARAMETERS, 'n_estimators': 400, 'max_depth': 8,
                   'learning_rate': 0.1, 'num_leaves': 15, 'min_child_samples': 40}
XGBOOST_PARAMS = {**ms.XGBOOST_PARAMETERS, 'n_estimators': 500, 'max_depth': 6,
                  'learning_rate': 0.1, 'min_child_weight': 1}
if hpo['final_configuration']['family'] != 'LightGBM' or hpo['final_configuration']['parameters'] != LIGHTGBM_PARAMS:
    raise ValueError('Primary LightGBM parameters differ from completed HPO')
if hpo['ensemble_candidate']['family'] != 'XGBRegressor' or hpo['ensemble_candidate']['parameters'] != XGBOOST_PARAMS:
    raise ValueError('Secondary XGBoost parameters differ from completed HPO')
for family in ('LightGBM', 'XGBRegressor'):
    prepared_train = ms.make_features(train, CONFIGURATION, family)
    prepared_forecast = ms.make_features(forecast, CONFIGURATION, family)
    if prepared_train.columns.tolist() != list(CALENDAR_WEATHER_FEATURES) or not prepared_train.dtypes.equals(prepared_forecast.dtypes):
        raise ValueError(f'Preprocessing contract changed: {family}')
display(pd.DataFrame([{'model': 'LightGBM', 'role': 'primary', 'features': len(CALENDAR_WEATHER_FEATURES),
                       'HPO pooled WAPE': hpo['final_configuration']['pooled_wape']},
                      {'model': 'XGBRegressor', 'role': 'secondary', 'features': len(CALENDAR_WEATHER_FEATURES),
                       'HPO pooled WAPE': hpo['ensemble_candidate']['pooled_wape']}]).round(6))

,model,role,features,HPO pooled WAPE
0,LightGBM,primary,12,0.150956
1,XGBRegressor,secondary,12,0.153481


## Final Training Jan–Oct

Обе модели обучаются **независимо** на всех 72 960 строках Jan–Oct, включая Sep–Oct. Никакого нового выбора по одному Sep–Oct внутри notebook нет: параметры уже зафиксированы HPO. LightGBM обучается с указанными categorical columns, XGBoost — на тех же категориях через native categorical preprocessing.

In [4]:
model_specs = {
    'lightgbm_primary': ('LightGBM', LGBMRegressor(**LIGHTGBM_PARAMS)),
    'xgboost_secondary': ('XGBRegressor', XGBRegressor(**XGBOOST_PARAMS)),
}
final_models, training_seconds = {}, {}
for name, (family, model) in model_specs.items():
    x_train = ms.make_features(train, CONFIGURATION, family)
    fit_options = {'categorical_feature': list(ms.CATEGORICAL)} if family == 'LightGBM' else {}
    start = perf_counter()
    model.fit(x_train, train['boardings'], **fit_options)
    training_seconds[name] = perf_counter() - start
    final_models[name] = model
display(pd.DataFrame([{'model': name, 'trained_rows': len(train),
                       'fit_seconds': training_seconds[name]}
                      for name in final_models]).round(3))

,model,trained_rows,fit_seconds
0,lightgbm_primary,72960,1.939
1,xgboost_secondary,72960,18.323


## Nov–Dec Inference

Для каждой модели прогнозируем **ровно строки исходного forecast CSV в том же порядке**. Используем тот же preprocessing, что в HPO, проверяем конечность raw predictions, затем применяем прежний postprocessing: `floor(max(raw, 0) + 0.5)`. Для route 5 итоговый prediction принудительно равен `0`: у него нет валидной положительной target history, а это действующий inference contract. Итоговые значения должны быть конечными, целыми и неотрицательными.

In [5]:
inferences, prediction_seconds = {}, {}
for name, (family, _) in model_specs.items():
    x_forecast = ms.make_features(forecast, CONFIGURATION, family)
    start = perf_counter()
    raw_prediction = final_models[name].predict(x_forecast)
    prediction_seconds[name] = perf_counter() - start
    if not np.isfinite(raw_prediction).all():
        raise ValueError(f'Non-finite raw predictions: {name}')
    prediction = np.floor(np.maximum(raw_prediction, 0) + 0.5).astype('int64')
    prediction[forecast['route'].to_numpy() == 5] = 0
    inference = forecast.loc[:, KEYS].copy()
    inference['prediction'] = prediction
    if len(inference) != 14_640 or inference.isna().any().any():
        raise ValueError(f'Incomplete inference: {name}')
    if not np.isfinite(inference['prediction']).all() or inference['prediction'].lt(0).any():
        raise ValueError(f'Invalid predictions: {name}')
    if not pd.api.types.is_integer_dtype(inference['prediction']):
        raise TypeError(f'Predictions are not integers: {name}')
    if not inference.loc[inference['route'].eq(5), 'prediction'].eq(0).all():
        raise ValueError(f'Route 5 policy failed: {name}')
    inferences[name] = inference
display(pd.DataFrame([{'model': name, 'prediction_rows': len(frame),
                       'predict_seconds': prediction_seconds[name],
                       'route_5_nonzero': int(frame.loc[frame.route.eq(5), 'prediction'].ne(0).sum())}
                      for name, frame in inferences.items()]).round(3))

,model,prediction_rows,predict_seconds,route_5_nonzero
0,lightgbm_primary,14640,0.270,0
1,xgboost_secondary,14640,0.351,0


## Nov–Dec Inference Diagnostics

Эти сводки проверяют масштаб и форму прогноза; они **не являются оценкой accuracy**, поскольку фактических посадок за Nov–Dec нет. Показываем общие суммы, Nov/Dec, будни/выходные, маршруты, нули и квантили. Разницу LightGBM и XGBoost рассматриваем как диагностический сигнал для будущего ensemble experiment, не смешивая прогнозы сейчас.

In [6]:
diagnostic = forecast.loc[:, KEYS + ['weekday']].copy()
diagnostic['month'] = diagnostic['date'].dt.month.map({11: 'Nov', 12: 'Dec'})
diagnostic['day_type'] = np.where(diagnostic['weekday'].lt(5), 'weekday', 'weekend')
for name, frame in inferences.items():
    diagnostic[name] = frame['prediction'].to_numpy()

distribution_rows = []
for name in inferences:
    values = diagnostic[name]
    distribution_rows.append({'model': name, 'total': int(values.sum()),
                              'zero_count': int(values.eq(0).sum()),
                              'median': float(values.median()),
                              'p95': float(values.quantile(0.95)),
                              'p99': float(values.quantile(0.99)),
                              'max': int(values.max())})
display(pd.DataFrame(distribution_rows).round(2))
display(diagnostic.groupby('month', sort=False)[list(inferences)].sum().rename_axis('month'))
display(diagnostic.groupby('day_type', sort=False)[list(inferences)].sum().rename_axis('day_type'))
display(diagnostic.groupby('route', sort=True)[list(inferences)].sum().rename_axis('route'))

diagnostic['difference'] = diagnostic['lightgbm_primary'] - diagnostic['xgboost_secondary']
diagnostic['absolute_difference'] = diagnostic['difference'].abs()
display(pd.DataFrame([{'sum_difference': int(diagnostic['difference'].sum()),
                       'mean_absolute_difference': float(diagnostic['absolute_difference'].mean()),
                       'median_absolute_difference': float(diagnostic['absolute_difference'].median()),
                       'p95_absolute_difference': float(diagnostic['absolute_difference'].quantile(0.95)),
                       'max_absolute_difference': int(diagnostic['absolute_difference'].max())}]).round(2))
display(diagnostic.sort_values(['absolute_difference', 'route', 'date', 'hour'],
                               ascending=[False, True, True, True])[
    ['route', 'date', 'hour', 'lightgbm_primary', 'xgboost_secondary', 'difference', 'absolute_difference']
].head(10).assign(date=lambda frame: frame.date.dt.strftime('%Y-%m-%d')).reset_index(drop=True))

,model,total,zero_count,median,p95,p99,max
0,lightgbm_primary,12616787,3039,507.0,2934.05,3863.0,5246
1,xgboost_secondary,12666464,2835,503.0,2949.00,3955.0,5348


,lightgbm_primary,xgboost_secondary
month,,
Nov,6141027,6200475
Dec,6475760,6465989


,lightgbm_primary,xgboost_secondary
day_type,,
weekend,2545522,2573059
weekday,10071265,10093405


,lightgbm_primary,xgboost_secondary
route,,
1,1080977,1084265
5,0,0
7,1419450,1433787
11,1871565,1869091
12,1953638,1959002
17,2955677,2984654
25,418006,411066
26,1045974,1041839
28,595598,595334


,sum_difference,mean_absolute_difference,median_absolute_difference,p95_absolute_difference,max_absolute_difference
0,-49677,19.33,8.0,69.0,1156


,route,date,hour,lightgbm_primary,xgboost_secondary,difference,absolute_difference
0,17,2025-11-04,8,2541,1385,1156,1156
1,17,2025-12-31,8,2646,1564,1082,1082
2,12,2025-11-01,7,1878,1272,606,606
3,17,2025-11-04,18,3127,2522,605,605
4,17,2025-11-01,7,2588,2044,544,544
5,17,2025-12-31,18,3237,2723,514,514
6,17,2025-11-08,16,2977,3471,-494,494
7,17,2025-11-15,16,2977,3471,-494,494
8,17,2025-11-22,16,2977,3471,-494,494
9,17,2025-11-29,16,2977,3471,-494,494


## Submission

Сохраняем два независимых UTF-8 CSV с разделителем `;` и схемой `route;date;hour;prediction`. **Основной Pilot v4 submission — `pilot_model_v4_lightgbm_submission.csv`**. Файл XGBoost — secondary candidate для будущей проверки ensemble; blending сейчас не выполняется. Порядок строк берётся из forecast input без сортировки или реконструкции grid.

In [7]:
for name, inference in inferences.items():
    submission = inference.loc[:, ['route', 'date', 'hour', 'prediction']].copy()
    submission['date'] = submission['date'].dt.strftime('%Y-%m-%d')
    submission.to_csv(SUBMISSION_PATHS[name], sep=';', index=False, encoding='utf-8')
display(pd.DataFrame([{'role': name, 'path': path.relative_to(ROOT).as_posix(),
                       'rows': len(inferences[name])}
                      for name, path in SUBMISSION_PATHS.items()]))

,role,path,rows
0,lightgbm_primary,data/submissions/pilot_model_v4_lightgbm_submi...,14640
1,xgboost_secondary,data/submissions/pilot_model_v4_xgboost_submis...,14640


## Final Sanity Checks

После сохранения повторно читаем **оба** CSV. Проверяем точную схему и порядок ключей относительно forecast input и организаторского template, 14 640 строк, 10 маршрутов, 24 часа, весь период 1 ноября — 31 декабря, отсутствие дублей и NaN, конечные неотрицательные целые predictions и полный ноль route 5. Эти проверки защищают формат submission, но не оценивают точность будущего прогноза.

In [8]:
expected_keys = forecast[KEYS].copy()
expected_keys['date'] = expected_keys['date'].dt.strftime('%Y-%m-%d')
expected_keys = expected_keys.reset_index(drop=True)
template = pd.read_csv(DATA_ROOT / 'raw' / 'test_submission.csv', sep=';', encoding='utf-8')
if not template[KEYS].equals(expected_keys):
    raise ValueError('Forecast keys/order differ from submission template')
for name, path in SUBMISSION_PATHS.items():
    saved = pd.read_csv(path, sep=';', encoding='utf-8')
    if saved.columns.tolist() != ['route', 'date', 'hour', 'prediction']:
        raise ValueError(f'Saved submission columns or order differ: {name}')
    if len(saved) != 14_640 or saved[KEYS].duplicated().any():
        raise ValueError(f'Saved submission row count or keys are invalid: {name}')
    if saved.isna().any().any():
        raise ValueError(f'Saved submission contains NaN: {name}')
    if set(saved['route']) != set(ROUTES) or set(saved['hour']) != set(range(24)):
        raise ValueError(f'Saved submission routes or hours differ: {name}')
    if set(saved['date']) != set(pd.date_range('2025-11-01', '2025-12-31').strftime('%Y-%m-%d')):
        raise ValueError(f'Saved submission period differs: {name}')
    if not np.isfinite(saved['prediction']).all():
        raise ValueError(f'Saved predictions are non-finite: {name}')
    if saved['prediction'].lt(0).any() or not pd.api.types.is_integer_dtype(saved['prediction']):
        raise ValueError(f'Saved predictions must be non-negative integers: {name}')
    if not saved.loc[saved['route'].eq(5), 'prediction'].eq(0).all():
        raise ValueError(f'Saved route 5 predictions must be zero: {name}')
    if not saved[KEYS].equals(expected_keys):
        raise ValueError(f'Saved keys or row order differ from forecast input: {name}')
    print('Final sanity checks passed:', name, '14,640 Nov-Dec rows')
if 'boardings' in forecast or set(WEATHER_FEATURES) != {'temperature_bucket'}:
    raise ValueError('Target or weather feature contract changed')
display(pd.DataFrame([{'model': name, 'status': 'passed', 'rows': 14_640,
                       'routes': 10, 'hours': 24, 'route_5_nonzero': 0}
                      for name in SUBMISSION_PATHS]))

Final sanity checks passed: lightgbm_primary 14,640 Nov-Dec rows
Final sanity checks passed: xgboost_secondary 14,640 Nov-Dec rows


,model,status,rows,routes,hours,route_5_nonzero
0,lightgbm_primary,passed,14640,10,24,0
1,xgboost_secondary,passed,14640,10,24,0


## Итог Pilot v4

Оба final fits обучены на Jan–Oct и создали отдельные Nov–Dec submissions. Tuned LightGBM + `C+W` остаётся основной Pilot v4 конфигурацией, tuned XGBoost + `C+W` — secondary ensemble candidate. Accepted feature registry не изменён; ensemble и проверка Nov–Dec accuracy на этом этапе не выполнялись.